# Capital Management：Kelly 准则

对应 Hilpisch《Python for Algorithmic Trading》第 10 章中的 Capital Management 一节（仅此小节）。
体例对齐同目录其他笔记：正文中文；代码可运行；书中关键输出以注释或预置 stdout 标出。


## 0. 问题

算法交易的一个核心问题：在总本金给定时，一条策略应投入多少资金。多数个人与机构会同意，长期财富最大化是合理目标。Edward Thorp 据此将 Kelly 准则引入投资（见 Rotando and Thorp, 1992）。在策略收益统计特征已知时，Kelly 给出每期应投入可用资金的最优比例 $f^*$。


## 1. Capital Management：Kelly 准则

### 1.1 二项设定下的 Kelly（掷硬币）

经典引入方式是掷硬币或更一般的二项博弈（仅两种结果）。设定如下：

- 赌徒与「资金无限」的庄家对赌。
- 正面概率满足 $\frac{1}{2} < p < 1$，反面概率 $q = 1 - p < \frac{1}{2}$。
- 每局可下任意 $b > 0$：猜对赢得等额 $b$，猜错输掉全部 $b$。
- 因 $p > q$，理性赌徒押正面。

单局期望收益：

$$
\mathbb{E}[B] = p\cdot b - q\cdot b = (p - q)\cdot b > 0
$$

风险中性、资金无限的赌徒会想把本金全部押上，以最大化单期期望。但市场交易是重复博弈，不是一锤子买卖。

记第 $i$ 日下注 $b_i$，初始本金 $c_0$。重复 $n$ 次后的期望为：

$$
\mathbb{E}[B_n] = c_0 + \sum_{i=1}^{n}(p - q)\cdot b_i
$$

若每期都押上全部可用资金 $b_i = c_{i-1}$，一次失败即可破产。因此「全押」并不最大化长期财富；「一点不押」则浪费有利可图的赌局。Kelly 要求：找出每期应投入可用资金的最优比例 $f^*$。


设 $n = h + t$（$h$ 次正面，$t$ 次反面）。$n$ 局后本金为：

$$
c_n = c_0 \cdot (1+f)^{h} \cdot (1-f)^{t}
$$

长期财富最大化等价于最大化每局平均几何增长率：

$$
r_g = \log\left(\frac{c_n}{c_0}\right)^{1/n}
= \frac{h}{n}\log(1+f) + \frac{t}{n}\log(1-f)
$$

取期望（$\mathbb{E}[h]=np$，$\mathbb{E}[t]=nq$）得到目标函数：

$$
G(f) = p\log(1+f) + q\log(1-f)
$$

一阶条件 $G'(f)=0$ 给出：

$$
f^* = p - q
$$

例如 $p=0.55$，则 $f^*=0.55-0.45=0.1$，即每期投入可用资金的 10%。


### 1.2 二项设定：仿真

先导入与绘图配置。


In [ ]:
import math
import numpy as np
import pandas as pd
import datetime as dt
from pylab import plt, mpl

np.random.seed(1000)
plt.style.use("seaborn-v0_8")
mpl.rcParams["savefig.dpi"] = 300
mpl.rcParams["font.family"] = "serif"
plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei", "SimHei", "PingFang SC", "Noto Sans CJK SC", "Arial Unicode MS"
]
plt.rcParams["axes.unicode_minus"] = False


固定正面概率，按 Kelly 计算 $f^*$；再设定仿真规模（50 条路径，每条 100 次试验）。


In [ ]:
p = 0.55                          # ① 正面概率
f = p - (1 - p)                   # ② Kelly 最优比例 f* = p - q
print("f* =", f)                  # 书中 Out[5] ≈ 0.1

I = 50                            # ③ 仿真路径条数
n = 100                           # ④ 每条路径试验次数


f* = 0.10000000000000009


`run_simulation(f)`：在二维数组中记录资本路径。初始资本 100；每期掷硬币，赢则资本乘 $(1+f)$，输则乘 $(1-f)$。


In [ ]:
def run_simulation(f):
    c = np.zeros((n, I))          # 存放仿真结果
    c[0] = 100                    # 初始资本
    for i in range(I):            # 外层：第 i 条路径
        for t in range(1, n):     # 内层：第 t 期
            o = np.random.binomial(1, p)  # 掷硬币：1=正面，0=反面
            if o > 0:
                c[t, i] = (1 + f) * c[t - 1, i]   # 赢：加上赢利比例
            else:
                c[t, i] = (1 - f) * c[t - 1, i]   # 输：扣掉下注比例
    return c


c_1 = run_simulation(f)
print(c_1.round(2))              # 书中 Out[10]：形状 (100, 50)


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(c_1, "b", lw=0.5)                 # ① 50 条路径
plt.plot(c_1.mean(axis=1), "r", lw=2.5)    # ② 红色：路径均值
plt.title("Figure 10-1：50 条路径 × 100 次试验（红线=均值）")
plt.xlabel("trial")
plt.ylabel("capital")
plt.show()


再比较不同 $f$。过小则平均增长偏慢；过大（如 0.5）波动显著上升，期末均值甚至可能更差。见图 10-2。


In [ ]:
c_2 = run_simulation(0.05)   # ① f = 0.05（偏保守）
c_3 = run_simulation(0.25)   # ② f = 0.25（偏激进）
c_4 = run_simulation(0.5)    # ③ f = 0.5（过高）

plt.figure(figsize=(10, 6))
plt.plot(c_1.mean(axis=1), "r", label="$f^*=0.1$")
plt.plot(c_2.mean(axis=1), "b", label="$f=0.05$")
plt.plot(c_3.mean(axis=1), "y", label="$f=0.25$")
plt.plot(c_4.mean(axis=1), "m", label="$f=0.5$")
plt.legend(loc=0)
plt.title("Figure 10-2：不同 f 下资本路径的均值")
plt.show()


## 2. 股票与股指上的 Kelly

现改为更接近股市的二项设定：一年后股价指数相对今日只取两个可能值（Hung, 2010）：

$$
P\!\left(r_S=\mu+\sigma\right)=P\!\left(r_S=\mu-\sigma\right)=\frac{1}{2}
$$

其中 $\mu=\mathbb{E}[r_S]>0$ 为年化期望收益，$\sigma>0$ 为收益波动率。一年后可用资金：

$$
c(f)=c_0\cdot\bigl(1+(1-f)\cdot r + f\cdot r_S\bigr)
$$

$r$ 为未投资金的无风险利率。目标仍是最大化几何增长率 $G(f)=\mathbb{E}\log(c(f)/c_0)$。

把一年拆成 $n$ 个交易日，波动按 $\sqrt{n}$ 缩放后，在连续交易极限下可得：

$$
G_\infty(f)=r+(\mu-r)f-\frac{\sigma^2}{2}f^2
$$

一阶条件给出最优比例：

$$
f^*=\frac{\mu-r}{\sigma^2}
$$

即超额收益除以收益方差。形式接近夏普比率，但分母是方差而非波动率，含义不同。


### 2.1 实例：被动做多标普 500

读取书中 EOD 数据，计算对数收益与年化 $\mu$、$\sigma$，再求 $f^*$。数据源为 Hilpisch 提供的 CSV；若网络不可用，请自行下载到本地后改路径。


In [ ]:
DATA_URL = "http://hilpisch.com/pyalgo_eikon_eod_data.csv"
# 本地备选：DATA_URL = "data/pyalgo_eikon_eod_data.csv"

raw = pd.read_csv(DATA_URL, index_col=0, parse_dates=True)
symbol = ".SPX"
data = pd.DataFrame(raw[symbol])
data["return"] = np.log(data / data.shift(1))
data.dropna(inplace=True)
data.tail()
# 书中 Out[21] 末几行形如：
# 2019-12-31  3230.78  0.002942


In [ ]:
mu = data["return"].mean() * 252          # ① 年化收益
sigma = data["return"].std() * 252 ** 0.5  # ② 年化波动
r = 0.0                                    # ③ 无风险利率（示例中取 0）
f = (mu - r) / sigma ** 2                  # ④ Kelly 最优比例

print("mu    =", mu)       # 书中 ≈ 0.0999
print("sigma =", sigma)    # 书中 ≈ 0.1476
print("f*    =", f)        # 书中 ≈ 4.59
# 解读：每 1 美元权益，策略侧投入约 4.59 美元名义 → 杠杆约 4.59


mu    = 0.09992181916534204
sigma = 0.14761569775486563
f*    = 4.585590244019818


其他条件不变时：期望收益越高或波动越低，$f^*$ 越大，隐含杠杆越高。


### 2.2 按日调整杠杆的权益路径

为便于比较，初始权益设为 1，初始投入名义资本为 $1\cdot f^*$。每日按策略收益更新名义资本，再按固定杠杆把名义资本调回「当前权益 $\times f$」：亏则减名义、盈则加名义。图 10-3 对比指数本身与不同 $f$ 下的权益。


In [ ]:
equs = []


def kelly_strategy(f):
    """在 data 上按固定杠杆 f 模拟权益与名义资本路径。"""
    global equs
    equ = "equity_{:.2f}".format(f)
    equs.append(equ)
    cap = "capital_{:.2f}".format(f)
    data[equ] = 1                              # ① 初始权益 = 1
    data[cap] = data[equ] * f                  # ② 初始名义 = 1 * f
    for i, t in enumerate(data.index[1:]):
        t_1 = data.index[i]                    # ③ 上一交易日
        # ④ 名义资本随当日收益复利
        data.loc[t, cap] = data[cap].loc[t_1] * math.exp(data["return"].loc[t])
        # ⑤ 权益 = 旧权益 + 名义资本当日盈亏
        data.loc[t, equ] = (
            data[cap].loc[t] - data[cap].loc[t_1] + data[equ].loc[t_1]
        )
        # ⑥ 按新权益与固定杠杆重设名义资本
        data.loc[t, cap] = data[equ].loc[t] * f


kelly_strategy(f * 0.5)   # half Kelly
kelly_strategy(f * 0.66)  # 约 2/3 Kelly
kelly_strategy(f)         # full Kelly

print(data[equs].tail())
# 书中 Out[34] 末行量级约：equity_2.29≈6.58, equity_3.03≈9.50, equity_4.59≈14.01


In [ ]:
ax = data["return"].cumsum().apply(np.exp).plot(figsize=(10, 6))
data[equs].plot(ax=ax, legend=True)
ax.set_title("Figure 10-3：标普 500 累计表现 vs 不同 f 下的权益")
plt.show()


### 2.3 实务含义：Full Kelly 与 Half Kelly

图 10-3 中，full Kelly（约 4.59）权益路径波动很大，符合高杠杆直觉。实务上常不用 full Kelly，而用 half Kelly（约 2.3），以降低权益波动：

$$
\frac{1}{2}f^* \approx 2.3
$$

$f$ 越小，权益曲线通常越平滑，但长期几何增长也可能低于「接近最优」的选择。仓位是收益与回撤之间的显式权衡，而非越大越好。


## 3. 小结

| 设定 | 最优比例 |
|---|---|
| 二项赌局（赢输对称、概率 $p,q$） | $f^*=p-q$ |
| 股票/股指（连续极限） | $f^*=(\mu-r)/\sigma^2$ |

要点：

1. 单期期望最大化会推出“全押”，与长期财富目标冲突。
2. Kelly 最大化几何增长率，给出可计算的资金比例。
3. 股指被动多头示例中，$f^*>1$ 意味着杠杆；仿真显示 full Kelly 波动剧烈，实务常取一半或更低。
4. 实盘前，$\mu$、$\sigma$ 须来自稳健估计；估计误差会直接扭曲 $f^*$。


## 4. 参考

- Rotando, L. I. and Thorp, E. O. (1992). The Kelly criterion in blackjack, sports betting, and the stock market.  
- Hung, J. (2010)（书中股指二项设定参考文献）.  
- Yves Hilpisch, *Python for Algorithmic Trading*, Chapter 10, Capital Management.  
- 数据：`http://hilpisch.com/pyalgo_eikon_eod_data.csv`
